# RoPE (Rotary Positional Encoding) Graphs

This notebook visualizes how RoPE rotates embedding pairs by position and why attention similarity becomes a function of relative position.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def rope_inv_freq(d_model, theta_base=10000.0):
    if d_model % 2 != 0:
        raise ValueError('RoPE expects an even embedding dimension.')
    pair_idx = np.arange(0, d_model, 2, dtype=np.float64)
    return theta_base ** (-pair_idx / d_model)


def rope_angles(seq_len, d_model, theta_base=10000.0):
    pos = np.arange(seq_len, dtype=np.float64)[:, None]
    inv_freq = rope_inv_freq(d_model, theta_base)[None, :]
    return pos * inv_freq  # (seq_len, d_model // 2)


def apply_rope(x, positions, theta_base=10000.0):
    # x: (T, D), positions: (T,)
    t, d = x.shape
    if d % 2 != 0:
        raise ValueError('RoPE expects an even embedding dimension.')
    if positions.shape[0] != t:
        raise ValueError('positions length must match x.shape[0].')

    inv_freq = rope_inv_freq(d, theta_base)
    angles = positions[:, None].astype(np.float64) * inv_freq[None, :]
    c = np.cos(angles)
    s = np.sin(angles)

    x_even = x[:, 0::2]
    x_odd = x[:, 1::2]

    out = np.empty_like(x, dtype=np.float64)
    out[:, 0::2] = x_even * c - x_odd * s
    out[:, 1::2] = x_even * s + x_odd * c
    return out

In [ ]:
seq_len = 64
d_model = 64
theta_base = 10000.0

angles = rope_angles(seq_len, d_model, theta_base)

plt.figure(figsize=(12, 5))
plt.pcolormesh(angles, cmap='viridis')
plt.title('RoPE Angles: position x frequency pair')
plt.xlabel('Pair index (dimension/2)')
plt.ylabel('Token position')
plt.colorbar(label='angle (radians)')
plt.show()

# Visualize the trajectory of one 2D pair under RoPE rotation.
positions = np.arange(seq_len, dtype=np.float64)
seed_pair = np.array([1.0, 0.0], dtype=np.float64)
w0 = rope_inv_freq(d_model, theta_base)[0]
pair_xy = np.stack([
    np.cos(positions * w0) * seed_pair[0] - np.sin(positions * w0) * seed_pair[1],
    np.sin(positions * w0) * seed_pair[0] + np.cos(positions * w0) * seed_pair[1],
], axis=1)

plt.figure(figsize=(6, 6))
plt.plot(pair_xy[:, 0], pair_xy[:, 1], 'o-', markersize=3)
plt.axhline(0, color='gray', linewidth=0.8)
plt.axvline(0, color='gray', linewidth=0.8)
plt.title('RoPE rotation trajectory (first pair)')
plt.xlabel('dim 0 component')
plt.ylabel('dim 1 component')
plt.axis('equal')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Relative-position behavior: Q(p)·K(q) depends on (p - q).
rng = np.random.default_rng(0)
q0 = rng.normal(size=(d_model,)).astype(np.float64)
k0 = rng.normal(size=(d_model,)).astype(np.float64)

positions = np.arange(seq_len, dtype=np.float64)
Q = np.repeat(q0[None, :], seq_len, axis=0)
K = np.repeat(k0[None, :], seq_len, axis=0)
Q_rot = apply_rope(Q, positions, theta_base)
K_rot = apply_rope(K, positions, theta_base)
sim = Q_rot @ K_rot.T

plt.figure(figsize=(8, 6))
plt.pcolormesh(sim, cmap='coolwarm')
plt.title('RoPE similarity matrix: Q(p)·K(q)')
plt.xlabel('q position')
plt.ylabel('p position')
plt.colorbar(label='dot product')
plt.show()

# Collapse by relative offset to show Toeplitz structure.
offsets = np.arange(-(seq_len - 1), seq_len)
offset_means = np.array([np.mean(np.diag(sim, k=o)) for o in offsets])

plt.figure(figsize=(10, 4))
plt.plot(offsets, offset_means, marker='o', markersize=3)
plt.title('Average similarity vs relative position (p - q)')
plt.xlabel('relative offset')
plt.ylabel('mean dot product')
plt.grid(alpha=0.3)
plt.show()

## Direct APE vs RoPE comparison (pairwise probe)\n\nTo compare apples-to-apples, use the same angle grid and probe each 2D pair with `(1, 0)`.\nIn this setup, RoPE gives `(cos, sin)` per pair, while APE stores `(sin, cos)`, so we swap RoPE channels before comparing.

In [ ]:
# Build APE table from the same RoPE angle grid.
positions = np.arange(seq_len, dtype=np.float64)
theta = positions[:, None] * rope_inv_freq(d_model, theta_base)[None, :]

ape_table = np.empty((seq_len, d_model), dtype=np.float64)
ape_table[:, 0::2] = np.sin(theta)
ape_table[:, 1::2] = np.cos(theta)

# RoPE probe: each pair starts at (1, 0).
probe = np.zeros((seq_len, d_model), dtype=np.float64)
probe[:, 0::2] = 1.0
rope_probe = apply_rope(probe, positions, theta_base)

# Align RoPE pair channels to APE ordering: (sin, cos).
rope_aligned = np.empty_like(rope_probe)
rope_aligned[:, 0::2] = rope_probe[:, 1::2]
rope_aligned[:, 1::2] = rope_probe[:, 0::2]

diff = ape_table - rope_aligned
print('max |APE - aligned RoPE| =', np.max(np.abs(diff)))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

im0 = axes[0].pcolormesh(ape_table, cmap='viridis')
axes[0].set_title('APE table (sin, cos)')
axes[0].set_xlabel('Embedding dimension')
axes[0].set_ylabel('Token position')
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].pcolormesh(rope_aligned, cmap='viridis')
axes[1].set_title('RoPE probe, channel-aligned')
axes[1].set_xlabel('Embedding dimension')
axes[1].set_ylabel('Token position')
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].pcolormesh(diff, cmap='coolwarm')
axes[2].set_title('Difference: APE - aligned RoPE')
axes[2].set_xlabel('Embedding dimension')
axes[2].set_ylabel('Token position')
fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
# Single RoPE probe plot at 512 dimensions.
d_model_probe = 512
positions = np.arange(seq_len, dtype=np.float64)

probe_512 = np.zeros((seq_len, d_model_probe), dtype=np.float64)
probe_512[:, 0::2] = 1.0
rope_probe_512 = apply_rope(probe_512, positions, theta_base)

# Align pair channels to (sin, cos) ordering used in APE-style heatmaps.
rope_probe_512_aligned = np.empty_like(rope_probe_512)
rope_probe_512_aligned[:, 0::2] = rope_probe_512[:, 1::2]
rope_probe_512_aligned[:, 1::2] = rope_probe_512[:, 0::2]

plt.figure(figsize=(14, 5))
plt.pcolormesh(rope_probe_512_aligned, cmap='viridis')
plt.title('RoPE probe (channel-aligned), d_model=512')
plt.xlabel('Embedding dimension')
plt.ylabel('Token position')
plt.xlim((0, d_model_probe))
plt.ylim((seq_len, 0))
plt.colorbar(label='value')
plt.show()